In [ ]:
#!/usr/bin/env python3
# semantic_segmentation.py
import os
import argparse
from typing import Tuple, List

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
from torchvision.datasets import VOCSegmentation
from tqdm import tqdm


def parse_args() -> argparse.Namespace:
    """Parse command-line arguments."""
    p = argparse.ArgumentParser(
        description="Train ResNet-based semantic segmentation on Pascal VOC"
    )
    p.add_argument("--data-root", type=str, default="./data",
                   help="Path to VOC dataset root")
    p.add_argument("--batch-size", type=int, default=8,
                   help="Batch size for training and validation")
    p.add_argument("--epochs", type=int, default=100,
                   help="Number of training epochs")
    p.add_argument("--lr", type=float, default=1e-3,
                   help="Initial learning rate")
    p.add_argument("--weight-decay", type=float, default=1e-4,
                   help="Weight decay for optimizer")
    p.add_argument("--step-size", type=int, default=10,
                   help="StepLR scheduler step size")
    p.add_argument("--gamma", type=float, default=0.1,
                   help="StepLR scheduler gamma")
    p.add_argument("--num-workers", type=int, default=2,
                   help="Number of DataLoader workers")
    p.add_argument("--input-size", type=int, default=224,
                   help="Input image height/width")
    return p.parse_args()


def get_transforms(input_size: int) -> Tuple[transforms.Compose, transforms.Compose]:
    """Create both image and target transforms."""
    img_transform = transforms.Compose([
        transforms.Resize((input_size, input_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

    # PILToTensor yields a ByteTensor with raw pixel values [0..255].
    # Squeeze channel and convert to LongTensor for class indices.
    tgt_transform = transforms.Compose([
        transforms.Resize((input_size, input_size), interpolation=Image.NEAREST),
        transforms.PILToTensor(),
        transforms.Lambda(lambda x: x.squeeze(0).long())
    ])

    return img_transform, tgt_transform


def get_dataloaders(
    root: str,
    img_t: transforms.Compose,
    tgt_t: transforms.Compose,
    batch_size: int,
    num_workers: int
) -> Tuple[DataLoader, DataLoader]:
    """Load Pascal VOC train/val splits into DataLoaders."""
    train_ds = VOCSegmentation(
        root=root, year="2012", image_set="train", download=True,
        transform=img_t, target_transform=tgt_t
    )
    val_ds = VOCSegmentation(
        root=root, year="2012", image_set="val", download=True,
        transform=img_t, target_transform=tgt_t
    )

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True
    )
    return train_loader, val_loader


def create_colormap(num_classes: int = 21) -> np.ndarray:
    """Return Pascal VOC standard 21-color map (RGB)."""
    palette = [
        (0, 0, 0), (128, 0, 0), (0, 128, 0), (128, 128, 0), (0, 0, 128),
        (128, 0, 128), (0, 128, 128), (128, 128, 128), (64, 0, 0),
        (192, 0, 0), (64, 128, 0), (192, 128, 0), (64, 0, 128),
        (192, 0, 128), (64, 128, 128), (192, 128, 128), (0, 64, 0),
        (128, 64, 0), (0, 192, 0), (128, 192, 0), (0, 64, 128)
    ]
    cmap = np.zeros((num_classes, 3), dtype=np.uint8)
    for i, col in enumerate(palette):
        cmap[i] = col
    return cmap


def visualize_sample(
    image: torch.Tensor,
    mask: torch.Tensor,
    colormap: np.ndarray,
    mean: List[float],
    std: List[float]
) -> None:
    """Denormalize image, map mask to colors, and plot side by side."""
    # Denormalize
    mean_t = torch.tensor(mean).view(3, 1, 1)
    std_t = torch.tensor(std).view(3, 1, 1)
    img = image * std_t + mean_t
    img = torch.clamp(img, 0, 1).permute(1, 2, 0).cpu().numpy()

    # Map mask
    mask_np = mask.cpu().numpy()
    mask_np[mask_np == 255] = 0  # treat ignore as background for viz
    mask_colored = colormap[mask_np]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
    ax1.imshow(img)
    ax1.set_title("Image")
    ax1.axis("off")
    ax2.imshow(mask_colored)
    ax2.set_title("Mask")
    ax2.axis("off")
    plt.tight_layout()
    plt.show()


class ResNetSegmentation(nn.Module):
    """Lightweight segmentation decoder on top of ResNet18 backbone."""
    def __init__(self, num_classes: int = 21):
        super().__init__()
        resnet18 = torchvision.models.resnet18(pretrained=True)
        # drop avgpool + fc
        self.encoder = nn.Sequential(*list(resnet18.children())[:-2])

        self.decoder = nn.Sequential(
            # 7→14
            nn.ConvTranspose2d(512, 256, 3, 2, 1, 1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            # 14→28
            nn.ConvTranspose2d(256, 128, 3, 2, 1, 1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            # 28→56
            nn.ConvTranspose2d(128, 64, 3, 2, 1, 1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            # 56→112
            nn.ConvTranspose2d(64, 32, 3, 2, 1, 1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            # 112→224
            nn.ConvTranspose2d(32, 32, 3, 2, 1, 1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            # final logits
            nn.Conv2d(32, num_classes, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.encoder(x)
        out = self.decoder(feat)
        return out


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device
) -> float:
    """Run one training epoch; return average loss."""
    model.train()
    total_loss = 0.0
    loop = tqdm(loader, desc="Train", leave=False)
    for imgs, masks in loop:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    return total_loss / len(loader)


def validate_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device
) -> Tuple[float, float]:
    """Run one validation epoch; return (avg_loss, pixel_accuracy)."""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        loop = tqdm(loader, desc="Validate", leave=False)
        for imgs, masks in loop:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            loss = criterion(logits, masks)
            total_loss += loss.item()

            preds = logits.argmax(dim=1)
            mask_valid = masks != 255
            correct += (preds[mask_valid] == masks[mask_valid]).sum().item()
            total += mask_valid.sum().item()

            loop.set_postfix(loss=loss.item())
    avg_loss = total_loss / len(loader)
    pix_acc = correct / total if total > 0 else 0.0
    return avg_loss, pix_acc


def plot_metrics(
    train_losses: List[float],
    val_losses: List[float],
    val_accs: List[float]
) -> None:
    """Plot loss and accuracy curves."""
    epochs = range(1, len(train_losses) + 1)
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, 'b-', label="Train Loss")
    plt.plot(epochs, val_losses, 'r-', label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, val_accs, 'g-', label="Val Pixel Acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()


def main():
    args = parse_args()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    img_t, tgt_t = get_transforms(args.input_size)
    train_loader, val_loader = get_dataloaders(
        args.data_root, img_t, tgt_t,
        args.batch_size, args.num_workers
    )

    model = ResNetSegmentation(num_classes=21).to(device)
    print(f"Model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    criterion = nn.CrossEntropyLoss(ignore_index=255)
    optimizer = optim.Adam(
        model.parameters(),
        lr=args.lr,
        weight_decay=args.weight_decay
    )
    scheduler = optim.lr_scheduler.StepLR(
        optimizer,
        step_size=args.step_size,
        gamma=args.gamma
    )

    colormap = create_colormap()
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]

    best_acc = 0.0
    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(1, args.epochs + 1):
        print(f"\nEpoch {epoch}/{args.epochs}")
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)
        scheduler.step()

        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val   Loss: {val_loss:.4f}, Val Pixel Acc: {val_acc:.4f}")

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        # Checkpoint best
        if val_acc > best_acc:
            best_acc = val_acc
            ckpt_path = os.path.join("checkpoints", "best_model.pth")
            os.makedirs("checkpoints", exist_ok=True)
            torch.save(model.state_dict(), ckpt_path)
            print(f"  Saved best model (acc={best_acc:.4f}) → {ckpt_path}")

    # Final plots
    plot_metrics(history["train_loss"], history["val_loss"], history["val_acc"])

    # Visualize a few predictions
    print("\nVisualizing some validation samples:")
    model.eval()
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs).argmax(dim=1)
            for i in range(min(3, imgs.size(0))):
                visualize_sample(imgs[i], preds[i], colormap, mean, std)
            break  # only first batch


if __name__ == "__main__":
    main()